In [ ]:
# 0. 라이브러리 & 기본 설정 (기존과 동일)
import os
import gc
import json
import time
import random
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_cosine_schedule_with_warmup
)
from torch.optim import AdamW

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
    balanced_accuracy_score, matthews_corrcoef, roc_auc_score
)

from tqdm.auto import tqdm

# 전처리에 필요한 라이브러리
# !pip install soynlp emoji -q   # Kaggle에서 필요하면 실행
import emoji
from soynlp.normalizer import repeat_normalize

SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("사용 장치:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

MODEL_NAME = "beomi/kcbert-base"
NUM_LABELS = 3
MAX_LENGTH = 128
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
PATIENCE = 3
MIN_DELTA = 0.0001

OUTPUT_DIR = Path("/kaggle/working/kcbert_final_cleaned")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BEST_MODEL_DIR = OUTPUT_DIR / "best_model"
LAST_MODEL_DIR = OUTPUT_DIR / "last_model"

print("저장 경로:", OUTPUT_DIR)


In [ ]:
# 1. 전처리 함수 (핵심)
# 전각 → 반각 변환용
FULL2HALF = str.maketrans(
    "０１２３４５６７８９ＡＢＣＤＥＦＧＨＩＪＫＬＭＮＯＰＱＲＳＴＵＶＷＸＹＺａｂｃｄｅｆｇｈｉｊｋｌｍｎｏｐｑｒｓｔｕｖｗｘｙｚ！＂＃＄％＆＇（）＊＋，－．／：；＜＝＞？＠［＼］＾＿｀｛｜｝～　",
    "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz!\"#$%&'()*+,-./:;<=>?@[\\]^_`{|}~ "
)

emojis = "".join(emoji.EMOJI_DATA.keys())
pattern = re.compile(f"[^ .,?!/@$%~％·∼()\x00-\x7Fㄱ-힣{emojis}]+")
url_pattern = re.compile(
    r"https?://(www\.)?[-a-zA-Z0-9@:%._\+~#=]{1,256}\.[a-zA-Z0-9()]{1,6}\b([-a-zA-Z0-9()@:%_\+.~#?&//=]*)"
)

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    
    # 1. 전각 → 반각
    text = text.translate(FULL2HALF)
    
    # 2. URL을 [URL]로 치환 (피싱 신호 유지)
    text = url_pattern.sub(" [URL] ", text)
    
    # 3. 허용 문자 외 제거 (한글, 영어, 숫자, 기본 특수문자, 이모지 유지)
    text = pattern.sub(" ", text)
    
    # 4. 공백 정리
    text = re.sub(r"\s+", " ", text).strip()
    
    # 5. 반복 문자 정규화 (ㅋㅋㅋㅋ → ㅋㅋ, !!! → !! 등)
    text = repeat_normalize(text, num_repeats=2)
    
    return text

In [ ]:
# 2. 데이터 로드 & 표준화 + 전처리 적용
DATA_ROOT = Path("/kaggle/input/datasets/")

def find_dataset_file(filename):
    matches = list(DATA_ROOT.rglob(filename))
    if not matches:
        raise FileNotFoundError(f"{filename} 파일을 찾을 수 없습니다.")
    if len(matches) > 1:
        print(f"{filename} 후보:")
        for path in matches:
            print(path)
        raise ValueError(f"{filename} 파일이 여러 개 발견되었습니다.")
    return matches[0]

def load_csv(file_path):
    for encoding in ["utf-8", "cp949", "euc-kr"]:
        try:
            df = pd.read_csv(file_path, encoding=encoding)
            print(f"{file_path.name} 로드 성공: {encoding}")
            return df
        except UnicodeDecodeError:
            continue
    raise ValueError(f"인코딩 확인 필요: {file_path}")

def find_column(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None

def standardize_dataset(df, dataset_name, text_candidates, label_candidates, label_mapping):
    text_col = find_column(df, text_candidates)
    label_col = find_column(df, label_candidates)

    if text_col is None or label_col is None:
        raise ValueError(f"{dataset_name} 컬럼 확인 필요: {df.columns.tolist()}")

    result = df[[text_col, label_col]].copy()
    result.columns = ["content", "original_label"]

    # 전처리 적용
    result["content"] = result["content"].astype(str).map(clean_text)

    result["original_label"] = pd.to_numeric(result["original_label"], errors="coerce")
    result = result.dropna(subset=["original_label"])
    result["original_label"] = result["original_label"].astype(int)

    result["label"] = result["original_label"].map(label_mapping)
    result = result.dropna(subset=["label"])
    result["label"] = result["label"].astype(int)

    # 전처리 후 빈 문자열 제거
    result = result[
        (result["content"] != "") &
        (result["content"].str.lower() != "nan")
    ]

    result = result[["content", "label"]]
    print(f"{dataset_name}: {len(result):,}건 (전처리 후)")
    return result


# 데이터 로드
df1 = load_csv(find_dataset_file("dataset1.csv"))
df2 = load_csv(find_dataset_file("dataset2.csv"))
df3 = load_csv(find_dataset_file("dataset3.csv"))
df4 = load_csv(find_dataset_file("dataset4.csv"))

data1 = standardize_dataset(
    df1, "dataset1",
    ["content", "text", "message", "sms", "내용", "문자"],
    ["label", "class", "target", "라벨"],
    {0: 0, 1: 1}
)

data2 = standardize_dataset(
    df2, "dataset2",
    ["content", "text", "message", "sms", "내용", "문자"],
    ["label", "class", "target", "라벨"],
    {0: 0, 1: 1}
)

data3 = standardize_dataset(
    df3, "dataset3",
    ["content", "text", "TEXT", "message", "sms", "내용", "문자"],
    ["class", "label", "LABEL", "target", "라벨"],
    {1: 0, 2: 2, 3: 0}
)

data4 = standardize_dataset(
    df4, "dataset4",
    ["content", "TEXT", "text", "message", "sms", "내용", "문자"],
    ["LABEL", "label", "class", "target", "라벨"],
    {0: 0, 1: 2}
)

# 통합 + 중복 제거
total_df = pd.concat([data1, data2, data3, data4], ignore_index=True)
total_df = total_df.drop_duplicates(subset=["content"]).reset_index(drop=True)
total_df["label"] = total_df["label"].astype(int)

print("\n전체 데이터 (전처리 + 중복제거 후):", len(total_df))
print("\n클래스 분포:")
print(total_df["label"].value_counts().sort_index())

assert set(total_df["label"].unique()) == {0, 1, 2}
assert total_df["content"].notna().all()
assert (total_df["content"].str.len() > 0).all()

In [ ]:
# 3. Train / Val / Test 분할 (기존과 동일)
train_df, temp_df = train_test_split(
    total_df, test_size=0.20, random_state=SEED, stratify=total_df["label"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED, stratify=temp_df["label"]
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

# 분할 데이터 저장 (전처리된 상태로 저장)
train_df.to_csv(OUTPUT_DIR / "train_split.csv", index=False, encoding="utf-8-sig")
val_df.to_csv(OUTPUT_DIR / "validation_split.csv", index=False, encoding="utf-8-sig")
test_df.to_csv(OUTPUT_DIR / "test_split.csv", index=False, encoding="utf-8-sig")


In [ ]:
# 4. Dataset / DataLoader (기존과 동일)
class TextClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding=False
        )
        encoding["labels"] = int(self.labels[idx])
        return encoding


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding=True)

train_dataset = TextClassificationDataset(train_df["content"], train_df["label"], tokenizer, MAX_LENGTH)
val_dataset   = TextClassificationDataset(val_df["content"],   val_df["label"],   tokenizer, MAX_LENGTH)
test_dataset  = TextClassificationDataset(test_df["content"],  test_df["label"],  tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=data_collator, num_workers=2, pin_memory=torch.cuda.is_available())
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE * 2, shuffle=False,
                          collate_fn=data_collator, num_workers=2, pin_memory=torch.cuda.is_available())
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE * 2, shuffle=False,
                          collate_fn=data_collator, num_workers=2, pin_memory=torch.cuda.is_available())

print("Train 배치 수:", len(train_loader))
print("Validation 배치 수:", len(val_loader))
print("Test 배치 수:", len(test_loader))



In [ ]:
# 5. 모델, 가중치, Optimizer, Scheduler (기존과 동일)
class_counts = train_df["label"].value_counts().sort_index().values
class_weights = (len(train_df) / (NUM_LABELS * class_counts)) ** 0.5
class_weights = class_weights / class_weights.mean()
class_weights = torch.tensor(class_weights, dtype=torch.float32, device=device)

print("클래스별 데이터 수:", class_counts)
print("클래스 가중치:", class_weights.cpu().numpy())

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
model.config.id2label = {0: "정상", 1: "유해 표현", 2: "피싱·광고"}
model.config.label2id = {"정상": 0, "유해 표현": 1, "피싱·광고": 2}
model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

total_training_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_training_steps * 0.1)
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_training_steps
)

USE_AMP = device.type == "cuda"
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

print("전체 학습 스텝:", total_training_steps)
print("Warmup 스텝:", warmup_steps)
print("Mixed Precision:", USE_AMP)


In [ ]:
# 6. 평가 함수 & 학습 루프 (기존 코드 그대로 사용 가능)
from tqdm.auto import tqdm

def evaluate_model(model, loader, criterion):

    model.eval()

    total_loss = 0
    y_true = []
    y_pred = []
    y_probs = []

    with torch.no_grad():

        for batch in tqdm(loader, desc="평가", leave=False):

            batch = {
                key: value.to(device)
                for key, value in batch.items()
            }

            labels = batch.pop("labels")

            with torch.cuda.amp.autocast(enabled=USE_AMP):

                outputs = model(**batch)

                loss = criterion(
                    outputs.logits,
                    labels
                )

            total_loss += loss.item()

            probabilities = torch.softmax(
                outputs.logits,
                dim=1
            )

            predictions = torch.argmax(
                probabilities,
                dim=1
            )

            y_true.extend(labels.cpu().numpy())
            y_pred.extend(predictions.cpu().numpy())
            y_probs.extend(probabilities.cpu().numpy())

    avg_loss = total_loss / len(loader)

    metrics = {
        "loss": avg_loss,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(
            y_true, y_pred,
            average="macro",
            zero_division=0
        ),
        "recall_macro": recall_score(
            y_true, y_pred,
            average="macro",
            zero_division=0
        ),
        "f1_macro": f1_score(
            y_true, y_pred,
            average="macro",
            zero_division=0
        ),
        "f1_weighted": f1_score(
            y_true, y_pred,
            average="weighted",
            zero_division=0
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_true, y_pred
        ),
        "mcc": matthews_corrcoef(y_true, y_pred)
    }

    report = classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2],
        target_names=["정상", "유해 표현", "피싱·광고"],
        output_dict=True,
        zero_division=0
    )

    return metrics, y_true, y_pred, y_probs, report


# -------------------------
# 학습 기록 초기화
# -------------------------

history = []

best_val_f1 = -1
best_epoch = 0
early_stop_counter = 0

training_start = time.time()

# 토크나이저도 모델 폴더에 저장
tokenizer.save_pretrained(BEST_MODEL_DIR)
tokenizer.save_pretrained(LAST_MODEL_DIR)

# Training Loop
for epoch in range(EPOCHS):

    epoch_start = time.time()

    model.train()

    total_train_loss = 0

    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS}"
    )

    for batch in progress_bar:

        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        labels = batch.pop("labels")

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=USE_AMP):

            outputs = model(**batch)

            loss = criterion(
                outputs.logits,
                labels
            )

        scaler.scale(loss).backward()

        # Gradient Clipping
        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        scaler.step(optimizer)
        scaler.update()

        scheduler.step()

        total_train_loss += loss.item()

        progress_bar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "lr": f"{scheduler.get_last_lr()[0]:.2e}"
        })

    train_loss = total_train_loss / len(train_loader)

    # -------------------------
    # Validation
    # -------------------------

    val_metrics, val_true, val_pred, val_probs, val_report = (
        evaluate_model(model, val_loader, criterion)
    )

    val_loss = val_metrics["loss"]
    val_f1 = val_metrics["f1_macro"]

    epoch_time = time.time() - epoch_start

    current_lr = scheduler.get_last_lr()[0]

    # 클래스별 지표 기록
    record = {
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_accuracy": val_metrics["accuracy"],
        "val_precision_macro": val_metrics["precision_macro"],
        "val_recall_macro": val_metrics["recall_macro"],
        "val_f1_macro": val_f1,
        "val_f1_weighted": val_metrics["f1_weighted"],
        "val_balanced_accuracy": val_metrics["balanced_accuracy"],
        "val_mcc": val_metrics["mcc"],
        "val_normal_f1": val_report["정상"]["f1-score"],
        "val_harmful_f1": val_report["유해 표현"]["f1-score"],
        "val_phishing_f1": val_report["피싱·광고"]["f1-score"],
        "learning_rate": current_lr,
        "epoch_time_sec": epoch_time
    }

    history.append(record)

    # 매 Epoch 기록 저장
    history_df = pd.DataFrame(history)

    history_df.to_csv(
        OUTPUT_DIR / "training_history.csv",
        index=False,
        encoding="utf-8-sig"
    )

    with open(
        OUTPUT_DIR / "training_history.json",
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            history,
            f,
            ensure_ascii=False,
            indent=4
        )

    print(f"\nEpoch {epoch + 1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")
    print(f"Val Accuracy: {val_metrics['accuracy']:.4f}")
    print(f"Val Macro F1: {val_f1:.4f}")
    print(f"Val Normal F1: {record['val_normal_f1']:.4f}")
    print(f"Val Harmful F1: {record['val_harmful_f1']:.4f}")
    print(f"Val Phishing F1: {record['val_phishing_f1']:.4f}")
    print(f"Epoch Time: {epoch_time:.1f} sec")

    # -------------------------
    # 마지막 모델 저장
    # -------------------------

    model.save_pretrained(LAST_MODEL_DIR)

    # -------------------------
    # Early Stopping 및 최고 모델 저장
    # -------------------------

    if val_f1 > best_val_f1 + MIN_DELTA:

        best_val_f1 = val_f1
        best_epoch = epoch + 1
        early_stop_counter = 0

        model.save_pretrained(BEST_MODEL_DIR)

        print("최고 모델 갱신 완료!")

        with open(
            OUTPUT_DIR / "best_model_info.json",
            "w",
            encoding="utf-8"
        ) as f:
            json.dump(
                {
                    "best_epoch": best_epoch,
                    "best_val_f1": best_val_f1,
                    "best_val_loss": val_loss
                },
                f,
                ensure_ascii=False,
                indent=4
            )

    else:

        early_stop_counter += 1

        print(
            f"Validation Macro F1 개선 없음: "
            f"{early_stop_counter}/{PATIENCE}"
        )

    # -------------------------
    # Early Stopping
    # -------------------------

    if early_stop_counter >= PATIENCE:

        print("\nEarly Stopping 발동!")
        print("최적 Epoch:", best_epoch)
        print("최고 Validation Macro F1:", best_val_f1)

        break

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


total_training_time = time.time() - training_start

print("\n========== 학습 종료 ==========")
print("최적 Epoch:", best_epoch)
print("최고 Validation Macro F1:", round(best_val_f1, 4))
print("전체 학습 시간:", round(total_training_time / 60, 2), "분")
print("최적 모델 저장 위치:", BEST_MODEL_DIR)

In [ ]:
# 최적 모델 불러오기
best_tokenizer = AutoTokenizer.from_pretrained(
    BEST_MODEL_DIR
)

best_model = AutoModelForSequenceClassification.from_pretrained(
    BEST_MODEL_DIR
).to(device)

best_model.eval()

print("최적 모델 Epoch:", best_epoch)
print("최적 Validation Macro F1:", best_val_f1)

In [ ]:
# Test 평가
test_start = time.time()

test_metrics, test_true, test_pred, test_probs, test_report = (
    evaluate_model(
        best_model,
        test_loader,
        criterion
    )
)

test_time = time.time() - test_start

# ROC AUC 추가 계산
try:
    test_auc = roc_auc_score(
        test_true,
        test_probs,
        multi_class="ovr",
        average="macro"
    )
except ValueError:
    test_auc = None

test_metrics["roc_auc_ovr_macro"] = test_auc
test_metrics["test_time_sec"] = test_time

print("\n========== 최종 Test 결과 ==========")

for key, value in test_metrics.items():
    print(f"{key}: {value}")

print("\n========== 클래스별 평가 결과 ==========")

print(
    classification_report(
        test_true,
        test_pred,
        labels=[0, 1, 2],
        target_names=[
            "정상",
            "유해 표현",
            "피싱·광고"
        ],
        zero_division=0
    )
)


In [ ]:
# 최종 평가 결과 저장
final_result = {
    "model": MODEL_NAME,
    "best_epoch": best_epoch,
    "best_validation_macro_f1": float(best_val_f1),
    "total_training_time_sec": float(total_training_time),
    "test_metrics": {
        key: (
            float(value) if value is not None else None
        )
        for key, value in test_metrics.items()
    },
    "classification_report": test_report,
    "data_sizes": {
        "train": len(train_df),
        "validation": len(val_df),
        "test": len(test_df)
    },
    "training_config": {
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "max_length": MAX_LENGTH,
        "early_stopping_patience": PATIENCE,
        "seed": SEED,
        "class_weights": class_weights.cpu().tolist()
    }
}

with open(
    OUTPUT_DIR / "final_test_results.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_result,
        f,
        ensure_ascii=False,
        indent=4
    )


# 예측 결과 저장

prediction_df = pd.DataFrame({
    "content": test_df["content"].values,
    "actual_label": test_true,
    "predicted_label": test_pred,
    "actual_class": [
        ["정상", "유해 표현", "피싱·광고"][x]
        for x in test_true
    ],
    "predicted_class": [
        ["정상", "유해 표현", "피싱·광고"][x]
        for x in test_pred
    ],
    "correct": [
        a == b for a, b in zip(test_true, test_pred)
    ],
    "prob_normal": np.array(test_probs)[:, 0],
    "prob_harmful": np.array(test_probs)[:, 1],
    "prob_phishing": np.array(test_probs)[:, 2]
})

prediction_df.to_csv(
    OUTPUT_DIR / "test_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)


# 오분류 데이터 저장

misclassified_df = prediction_df[
    prediction_df["correct"] == False
].copy()

misclassified_df.to_csv(
    OUTPUT_DIR / "test_misclassified.csv",
    index=False,
    encoding="utf-8-sig"
)

print("최종 평가 결과 저장 완료")
print("전체 Test:", len(prediction_df))
print("정답:", prediction_df["correct"].sum())
print("오분류:", len(misclassified_df))


from pathlib import Path

OUTPUT_DIR = Path("/kaggle/working/kcbert_final")
BEST_MODEL_DIR = OUTPUT_DIR / "best_model"
BEST_MODEL_DIR.mkdir(parents=True, exist_ok=True)

# 모델 + 토크나이저 저장
model.save_pretrained(BEST_MODEL_DIR)
tokenizer.save_pretrained(BEST_MODEL_DIR)

print("저장 완료:", BEST_MODEL_DIR)
print("저장된 파일 목록:")
for f in BEST_MODEL_DIR.rglob("*"):
    print(f" - {f}")

